# Numerical Differentiation

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 06.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/03_numerical_differentiation.ipynb)

---

## 🎯 Learning Objective

Compare finite-difference approximations to automatic differentiation (forward and reverse mode), and use finite differences to gradient-check an implementation.

---

## 📐 Theory

`02.04` derived backpropagation as reverse-mode automatic differentiation. This notebook fills
in the two things that discussion deferred: what finite differences are (the crude, first
tool anyone reaches for) and what *forward-mode* AD is (the other exact, non-symbolic option) —
and why reverse mode specifically is what deep learning needs.

### Finite differences

The most direct way to estimate $f'(x)$ numerically is to go back to the definition and use a
small but finite $h$ instead of a true limit:

$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad \text{(forward difference)}, \qquad
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad \text{(central difference)}$$

Central differences are more accurate (error shrinks like $O(h^2)$ instead of $O(h)$, since the
$O(h)$ error terms cancel by symmetry) but both share the same fundamental tension: shrinking
$h$ reduces *truncation* error (how well the finite difference approximates the true limit) but
increases *rounding* error (subtracting two very close floating-point numbers, recall `06.02`'s
catastrophic cancellation), so there's an optimal $h$ that's neither too big nor too small —
finite differences are an *approximation*, never exact.

### Automatic differentiation: exact, not approximate

**Automatic differentiation (AD)** computes derivatives that are exact up to floating-point
rounding — no truncation error at all — by mechanically applying the chain rule to the sequence
of elementary operations a program actually performs, rather than approximating a limit. AD
comes in two flavors, differing in which direction information flows through the computation:

**Forward mode** propagates a derivative *alongside* each intermediate value, in the same order
the original computation runs. A clean way to implement this: represent every number as a
**dual number** $a + b\varepsilon$, where $\varepsilon$ is a symbol satisfying $\varepsilon^2=0$
(not zero itself, but its square vanishes). Arithmetic on dual numbers automatically tracks
derivatives: $(a+b\varepsilon)(c+d\varepsilon) = ac + (ad+bc)\varepsilon$ (the $bd\varepsilon^2$
term vanishes), which is exactly the product rule ($\frac{d}{dx}(uv) = u'v + uv'$) if $b, d$ are
$u', v'$. Seed $x$ as $x_0 + 1\cdot\varepsilon$ and every subsequent dual number's $\varepsilon$
coefficient is automatically $df/dx$ at that point — no separate backward pass needed.

**Reverse mode** (recall `02.04`'s full derivation) instead runs a complete forward pass first,
then walks the recorded computation graph backward, propagating $\partial(\text{output})/\partial(\text{node})$
using the local-gradient recipe.

### Forward vs. reverse: why direction matters at scale

For $f:\mathbb{R}^n\to\mathbb{R}^m$: **forward mode** costs one pass per *input* dimension and
gets the derivative with respect to every output at once — efficient when $n$ is small, $m$ is
large. **Reverse mode** costs one pass per *output* dimension and gets the derivative with
respect to every input at once — efficient when $m$ is small, $n$ is large. A neural network's
loss function has $n$ = millions/billions of parameters and $m=1$ (a single scalar loss) — the
single most lopsided case imaginable, which is exactly why every deep learning framework uses
reverse mode (backprop) rather than forward mode: computing a billion forward-mode passes to
get one gradient would be absurd, while one reverse-mode pass gets the whole gradient at once.

### Gradient checking

Because finite differences and AD are computed by entirely different mechanisms, comparing
them is a standard way to catch bugs in a hand-written backward pass: if your analytic gradient
implementation disagrees with a central-difference estimate by more than the expected numerical
tolerance, you have a bug, not a rounding artifact. This is exactly the "verify by an
independent method" discipline used throughout this repo's own from-scratch implementations.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting finite-difference error against step size $h$ on a log-log scale reveals the classic
U-shape: too-large $h$ has truncation error, too-small $h$ has rounding error, and there's a
sweet spot in between.

In [ ]:
# f(x) = sin(x), f'(x) = cos(x). Sweep h and measure finite-difference error against the exact
# analytic derivative -- this reveals the truncation-vs-rounding tradeoff directly.
f = np.sin
f_prime_exact = np.cos
x0 = 1.0
true_deriv = f_prime_exact(x0)

hs = np.logspace(-16, 0, 200)
forward_errors = np.abs((f(x0 + hs) - f(x0)) / hs - true_deriv)
central_errors = np.abs((f(x0 + hs) - f(x0 - hs)) / (2 * hs) - true_deriv)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.loglog(hs, forward_errors, color="#C44E52", lw=2, label="forward difference (O(h) truncation)")
ax.loglog(hs, central_errors, color="#4C72B0", lw=2, label="central difference (O(h^2) truncation)")
ax.set_xlabel("step size h"); ax.set_ylabel("|error| vs. true derivative")
ax.set_title("Finite differences: too-large h truncates, too-small h loses precision\n(rounding error, recall 06.02)")
ax.legend()
plt.show()

best_h_central = hs[np.argmin(central_errors)]
best_h_forward = hs[np.argmin(forward_errors)]
print(f"Best step size for central differences: h ~ {best_h_central:.2e}, error = {central_errors.min():.2e}")
print(f"Best step size for forward differences: h ~ {best_h_forward:.2e}, error = {forward_errors.min():.2e}")
print(f"Central differences reach a smaller minimum error using a LARGER optimal h --")
print(f"its faster O(h^2) truncation convergence lets it stop shrinking h sooner, before rounding error dominates.")

## 🐍 Implementation from Scratch

We build a minimal forward-mode AD engine using dual numbers, verify it against both finite
differences and `torch.autograd`'s reverse mode on the same function, then use finite
differences to gradient-check a hand-written backward pass.

In [ ]:
class Dual:
    """a + b*epsilon, with epsilon^2 = 0 by definition. Arithmetic on these numbers
    automatically carries a derivative alongside a value -- no separate backward pass."""
    def __init__(self, real, dual=0.0):
        self.real, self.dual = real, dual

    def __add__(self, other):
        other = other if isinstance(other, Dual) else Dual(other, 0.0)
        return Dual(self.real + other.real, self.dual + other.dual)
    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Dual) else Dual(other, 0.0)
        # (a+be)(c+de) = ac + (ad+bc)e + bd*e^2 = ac + (ad+bc)e, since e^2=0 -- this IS the
        # product rule, derived purely from epsilon^2=0, not assumed separately
        return Dual(self.real * other.real, self.real * other.dual + self.dual * other.real)
    __rmul__ = __mul__

    def sin(self):
        return Dual(np.sin(self.real), np.cos(self.real) * self.dual)

# f(x) = x^2 + 3x, an easy case to check by hand: f'(x) = 2x + 3
def f_dual(x):
    return x * x + Dual(3.0) * x

x0 = Dual(2.0, 1.0)  # seed dual=1.0 to ask for df/dx directly
result = f_dual(x0)
print(f"Forward-mode AD:  f(2)={result.real}, f'(2)={result.dual}")

h = 1e-6
fd = (lambda x: x**2 + 3*x)
finite_diff = (fd(2.0 + h) - fd(2.0 - h)) / (2 * h)
print(f"Central finite diff: f'(2)~{finite_diff:.6f}")

xt = torch.tensor(2.0, requires_grad=True)
(xt * xt + 3 * xt).backward()
print(f"torch.autograd (reverse mode): f'(2)={xt.grad.item()}")
print(f"All three agree: {np.allclose([result.dual, finite_diff, xt.grad.item()], 7.0, atol=1e-4)}")

# A harder case with a nonlinearity, to confirm the chain rule composes correctly through .sin()
def g_dual(x):
    return (x * x).sin()

x1 = Dual(1.5, 1.0)
res = g_dual(x1)
analytic = 2 * 1.5 * np.cos(1.5**2)  # d/dx[sin(x^2)] = cos(x^2) * 2x, by hand
print(f"\nsin(x^2) at x=1.5:  forward-mode AD f'={res.dual:.6f}, hand-derived={analytic:.6f}")
print(f"Match: {np.isclose(res.dual, analytic)}")

# --- Gradient checking: catch a DELIBERATELY WRONG hand-written gradient ---
def my_function(x):
    return np.sum(x**3)

def my_gradient_CORRECT(x):
    return 3 * x**2

def my_gradient_BUGGY(x):
    return 2 * x**2   # deliberately wrong coefficient, simulating a real typo

def gradient_check(f, grad_fn, x, h=1e-5):
    numeric_grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus, x_minus = x.copy(), x.copy()
        x_plus[i] += h; x_minus[i] -= h
        numeric_grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    analytic_grad = grad_fn(x)
    max_diff = np.max(np.abs(numeric_grad - analytic_grad))
    return max_diff, numeric_grad, analytic_grad

x_test = np.array([1.0, 2.0, 3.0])
diff_correct, _, _ = gradient_check(my_function, my_gradient_CORRECT, x_test)
diff_buggy, _, _ = gradient_check(my_function, my_gradient_BUGGY, x_test)
print(f"\nGradient check, CORRECT implementation: max diff = {diff_correct:.2e}  (passes)")
print(f"Gradient check, BUGGY implementation:    max diff = {diff_buggy:.2e}  (FAILS -- caught the bug)")

## 🤖 Why This Matters for AI

`torch.autograd` is exactly the reverse-mode AD engine described above, scaled from scalars to
tensors and vectorized. When implementing a custom `torch.autograd.Function` with a hand-written
backward pass (common when writing a new, faster kernel for an existing operation), the standard
sanity check is exactly the gradient-checking technique above: PyTorch even ships it built in as
`torch.autograd.gradcheck`, which compares your analytic backward pass against finite differences
and raises an error if they disagree beyond a tolerance. Below, we use `torch.autograd.gradcheck`
directly on a custom autograd Function, first with a correct backward pass (passes) and then with
a deliberately introduced bug (fails) — confirming this isn't just a hand-rolled convenience but
a real, load-bearing tool used when extending PyTorch itself.

In [ ]:
class CubeCorrect(torch.autograd.Function):
    """A custom autograd Function for x^3, with a CORRECT hand-written backward pass."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 3

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 3 * x ** 2   # d/dx(x^3) = 3x^2

class CubeBuggy(torch.autograd.Function):
    """The SAME operation, but with a deliberately wrong coefficient in the backward pass --
    simulating exactly the kind of typo gradient checking exists to catch."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 3

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 2 * x ** 2   # BUG: should be 3*x**2, not 2*x**2

x_check = torch.randn(5, dtype=torch.float64, requires_grad=True)

correct_passes = torch.autograd.gradcheck(CubeCorrect.apply, (x_check,))
print(f"Correct backward pass: gradcheck passes = {correct_passes}")

try:
    torch.autograd.gradcheck(CubeBuggy.apply, (x_check,))
    print("Buggy backward pass: gradcheck passed (this should NOT happen)")
except RuntimeError:
    print("Buggy backward pass: gradcheck correctly RAISED an error -- the bug was caught")

print("\ntorch.autograd.gradcheck runs the exact same finite-difference comparison we built by")
print("hand above -- it's the production version of the same idea, used whenever PyTorch's own")
print("built-in operations need their hand-written backward passes verified.")

## 🏋️ Exercises

### Warm-up
1. By hand, compute $f'(2)$ for $f(x) = x^3 - 2x$ using both a forward difference with $h=0.1$
   and a central difference with the same $h$. Compare both against the exact analytic value
   $f'(2)=10$ and note which is closer.

### Practice
2. Add a `__pow__` method to `Dual` supporting `x ** n` for a positive integer $n$, with dual
   part $n\cdot a^{n-1}\cdot b$ (the power rule, derived the same way `__mul__` derives the
   product rule). Verify it reproduces `x*x*x`'s result for $n=3$ on a test point.

### Challenge
3. Add a `cos` method to `Dual` and use it to compute the derivative of
   $h(x) = \sin(x)\cos(x)$ via forward-mode AD. Then implement `gradient_check` for a
   *vector-valued* function (e.g. $\mathbf{f}(x,y) = (x^2y,\ xy^2)$, which has a $2\times2$
   Jacobian) by checking each output coordinate against each input coordinate separately.
   Compare the number of forward-mode passes vs. reverse-mode passes this would require, and
   explain which mode you'd prefer for this specific shape (recall the Theory section's
   forward-vs-reverse tradeoff).

---

## 📚 Further Reading
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)
- PyTorch documentation, [`torch.autograd.gradcheck`](https://pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html)

---

*Next notebook: [Numerical Linear Algebra](04_numerical_linear_algebra.ipynb)*